# Text De-identification

**Thursday Masking Day · Tilburg Multiscale Summerschool 2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WimPouw/TilburgMultiscaleSummerschool2026/blob/main/thursday-masking/notebooks/06_text_deidentification.ipynb)

---

Video gets the most attention — but behavioral research generates text too:  
interview transcripts, think-aloud protocols, chat logs, open survey responses.

Text carries the same re-identification risks as video (names, places, job titles,  
rare events) — and the same GDPR obligations.

This notebook shows how to **automatically de-identify text** using Named Entity  
Recognition (spaCy), then checks whether the semantic structure of the text  
survives — using a **word co-occurrence network** and a **word cloud**.

### What we cover

| # | Step | Tool |
|---|------|------|
| 1 | Load a transcript | plain text / your own file |
| 2 | NER-based replacement (PERSON, ORG, GPE → `[PERSON]` etc.) | spaCy |
| 3 | Pattern-based cleanup (email, phone, URL) | regex |
| 4 | Inspect what changed | diff view |
| 5 | Semantic network — does structure survive? | networkx |
| 6 | Word cloud before vs after | wordcloud |

### Connection to Thursday

- **Notebook 03** MaskBench — same question applied to video: does the signal survive masking?
- **Notebook 04** audio de-identification — voice is the spoken counterpart of this transcript
- **Notebook 05** archiving — a de-identified transcript is a companion sidecar to the masked video

## Setup

Run this cell first. On Colab it installs everything you need automatically.

In [ ]:
import sys
!{sys.executable} -m pip install -q spacy networkx wordcloud matplotlib
!{sys.executable} -m spacy download en_core_web_sm -q

import re
import json
import textwrap
import collections
from pathlib import Path

import spacy
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from wordcloud import WordCloud

plt.rcParams.update({
    'figure.facecolor': '#161616',
    'axes.facecolor':   '#262626',
    'axes.edgecolor':   '#525252',
    'axes.labelcolor':  '#c6c6c6',
    'xtick.color':      '#6f6f6f',
    'ytick.color':      '#6f6f6f',
    'text.color':       '#f4f4f4',
    'grid.color':       '#393939',
    'grid.linestyle':   '--',
})

nlp = spacy.load('en_core_web_sm')
print('Setup complete — spaCy model loaded.')

## Load a transcript

We use a **synthetic** interview transcript modelled on the triadic concept generation  
sessions from the Edelman (2011) dataset — names, institutions, and events are fictitious.

To use your own transcript, set `TRANSCRIPT_PATH` to a `.txt` file  
or replace `DEMO_TEXT` with your content.

In [ ]:
TRANSCRIPT_PATH = None   # set to Path('my_transcript.txt') to use your own file

DEMO_TEXT = """
Facilitator: Thanks for joining us today. Can you start by introducing yourself?

Participant A (Maria): Sure. I'm Maria Chen, a PhD student at Radboud University in the
cognitive neuroscience department. I work with Professor Jan de Vries on gesture studies.

Participant B (Thomas): I'm Thomas Bakker, also from Radboud. I study design cognition —
basically how teams like Philips or ASML make creative decisions under pressure.

Participant C (Leila): And I'm Leila Ahmadi, visiting from the University of Amsterdam.
My focus is on multimodal communication — speech, gesture, and gaze coordination.

Facilitator: Great. Today you'll work together to redesign this device. Before you start,
can you describe your first impressions?

Maria: It reminds me of something we saw at the CHI conference in Hamburg last year.
There was a project from MIT Media Lab that used a similar form factor.

Thomas: Yes, I saw that too — it was Yuki Tanaka's group, right? The one that got
the best paper award. I think we can take a completely different direction though.
If we contact Dr. Patel at the Eindhoven design lab, he might have some useful prototypes.

Leila: I agree. And I think the key insight from our Tuesday session — when we analysed
the gesture data from group 07 in the Edelman corpus — is that people use the physical
object as a shared reference. So the form matters as much as the function.

Maria: Right. My email is m.chen@radboudumc.nl if anyone wants to follow up.
Thomas, can you sketch that idea you mentioned? I'll document it.

Thomas: Sure — so imagine the device has three interaction zones, one per person in the
group. Like a triadic interface, literally. Each zone responds to that person's gestures
only — so there's no ambiguity about who is controlling what.
"""

if TRANSCRIPT_PATH:
    text = Path(TRANSCRIPT_PATH).read_text(encoding='utf-8')
else:
    text = DEMO_TEXT

print(f'Loaded transcript: {len(text.split())} words')
print()
print(textwrap.fill(text[:400].strip(), width=80), '...')

---
## Step 1 — Named Entity Recognition with spaCy

spaCy's `en_core_web_sm` model finds named entities automatically:  
people, organisations, locations, dates, and more.

We replace each entity with a **type tag** — `[PERSON]`, `[ORG]`, `[GPE]` —  
and optionally a **counter** so different people stay distinguishable: `[PERSON_1]`, `[PERSON_2]`.

In [ ]:
# Entity labels to replace and the tag each maps to
REPLACE_LABELS = {
    'PERSON':  '[PERSON]',
    'ORG':     '[ORG]',
    'GPE':     '[PLACE]',   # geo-political entity (countries, cities)
    'LOC':     '[PLACE]',
    'FAC':     '[PLACE]',
    'EVENT':   '[EVENT]',
    'WORK_OF_ART': '[WORK]',
}

KEEP_COUNTER = True   # True → [PERSON_1], [PERSON_2]; False → all become [PERSON]

def ner_deidentify(text: str) -> tuple[str, list[dict]]:
    """
    Replace named entities in `text` with type tags.
    Returns (de-identified text, list of replaced spans).
    """
    doc = nlp(text)
    counters = collections.defaultdict(int)
    seen     = {}   # canonical text → tag (so same name → same tag number)
    log      = []

    # Build replacement map (process longest spans first to avoid partial overlaps)
    replacements = []
    for ent in doc.ents:
        label = ent.label_
        if label not in REPLACE_LABELS:
            continue
        base_tag = REPLACE_LABELS[label]
        canon    = ent.text.strip().lower()
        if canon not in seen:
            counters[base_tag] += 1
            tag = f'{base_tag[:-1]}_{counters[base_tag]}]' if KEEP_COUNTER else base_tag
            seen[canon] = tag
        tag = seen[canon]
        replacements.append((ent.start_char, ent.end_char, ent.text, tag, label))
        log.append({'original': ent.text, 'tag': tag, 'label': label})

    # Apply replacements right-to-left so character offsets stay valid
    result = text
    for start, end, _, tag, _ in sorted(replacements, key=lambda x: x[0], reverse=True):
        result = result[:start] + tag + result[end:]

    return result, log


text_ner, ner_log = ner_deidentify(text)

print(f'Entities replaced: {len(ner_log)}')
print()
for entry in ner_log:
    print(f"  {entry['label']:<12} '{entry['original']}'  →  {entry['tag']}")

---
## Step 2 — Pattern-based cleanup

NER misses structured identifiers: **email addresses**, **phone numbers**,  
**URLs**, and **researcher IDs** (ORCID, etc.).  
Regex catches what the model misses.

In [ ]:
PATTERNS = [
    # email
    (r'[\w.+-]+@[\w-]+\.[\w.]+',                         '[EMAIL]'),
    # phone (various formats)
    (r'\+?[\d][\d\s\-().]{7,}[\d]',                     '[PHONE]'),
    # URL
    (r'https?://\S+|www\.\S+',                           '[URL]'),
    # ORCID
    (r'\d{4}-\d{4}-\d{4}-\d{3}[\dX]',                   '[ORCID]'),
    # Dutch BSN / student number (7–9 digits)
    (r'\b\d{7,9}\b',                                     '[ID]'),
]

def pattern_deidentify(text: str) -> tuple[str, list]:
    log = []
    for pattern, tag in PATTERNS:
        matches = re.findall(pattern, text)
        if matches:
            log.extend({'original': m, 'tag': tag} for m in matches)
        text = re.sub(pattern, tag, text)
    return text, log


text_deid, pattern_log = pattern_deidentify(text_ner)

print(f'Pattern matches replaced: {len(pattern_log)}')
for entry in pattern_log:
    print(f"  '{entry['original']}'  →  {entry['tag']}")

---
## Step 3 — Inspect the result

Side-by-side diff: original vs de-identified.
Replaced tokens are highlighted in amber.

In [ ]:
from IPython.display import HTML, display as ipy_display

def highlight_tags(text: str) -> str:
    """Wrap [TAG] tokens in an amber HTML span for notebook display."""
    return re.sub(
        r'(\[[A-Z_0-9]+\])',
        r'<mark style="background:#f1c21b;color:#161616;padding:1px 4px;'
        r'border-radius:2px;font-weight:bold">\1</mark>',
        text
    )

html = f"""
<div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;font-family:monospace;font-size:12px">
  <div style="background:#262626;padding:14px;border:1px solid #393939">
    <div style="color:#6f6f6f;margin-bottom:8px;font-size:10px;letter-spacing:.1em">ORIGINAL</div>
    <div style="color:#f4f4f4;white-space:pre-wrap">{text.strip()}</div>
  </div>
  <div style="background:#262626;padding:14px;border:1px solid #393939">
    <div style="color:#6f6f6f;margin-bottom:8px;font-size:10px;letter-spacing:.1em">DE-IDENTIFIED</div>
    <div style="color:#f4f4f4;white-space:pre-wrap">{highlight_tags(text_deid.strip())}</div>
  </div>
</div>
"""
ipy_display(HTML(html))

---
## Step 4 — Did de-identification break the semantic structure?

We build a **word co-occurrence network**: two content words are connected  
if they appear within a window of 5 words in the same sentence.

A researcher doing thematic or semantic analysis wants this network to survive  
masking — the *concepts* should still be there even if the *names* are gone.

In [ ]:
STOPWORDS = {
    'i', 'me', 'my', 'you', 'your', 'we', 'our', 'it', 'its', 'the', 'a', 'an',
    'is', 'was', 'are', 'were', 'be', 'been', 'have', 'has', 'had', 'do', 'does',
    'did', 'will', 'would', 'can', 'could', 'may', 'might', 'shall', 'should',
    'of', 'in', 'on', 'at', 'to', 'for', 'by', 'with', 'from', 'about', 'into',
    'that', 'this', 'these', 'those', 'and', 'or', 'but', 'if', 'so', 'as',
    'not', 'no', 'nor', 'than', 'too', 'very', 'just', 'also', 'there', 'here',
    'when', 'what', 'which', 'who', 'how', 'all', 'both', 'each', 'more', 'most',
    'up', 'out', 'over', 'then', 'same', 'right', 'like', 's', 'one', 'two',
    '[person]', '[org]', '[place]', '[event]', '[email]', '[phone]', '[url]',
    '[work]', '[id]', '[orcid]',
}

def build_cooccurrence(text: str, window: int = 5, min_freq: int = 1) -> nx.Graph:
    tokens = [
        t.lower().strip('.,;:!?"\'')
        for t in text.split()
        if len(t) > 2
    ]
    tokens = [t for t in tokens if t not in STOPWORDS and t.isalpha() or t.startswith('[')]

    G = nx.Graph()
    pairs = collections.Counter()
    for i, w in enumerate(tokens):
        for j in range(i + 1, min(i + window + 1, len(tokens))):
            pair = tuple(sorted([w, tokens[j]]))
            if pair[0] != pair[1]:
                pairs[pair] += 1

    for (u, v), weight in pairs.items():
        if weight >= min_freq:
            G.add_edge(u, v, weight=weight)

    return G


G_orig = build_cooccurrence(text)
G_deid = build_cooccurrence(text_deid)

print(f'Original network:       {G_orig.number_of_nodes()} nodes, {G_orig.number_of_edges()} edges')
print(f'De-identified network:  {G_deid.number_of_nodes()} nodes, {G_deid.number_of_edges()} edges')
print(f'Nodes preserved:        {len(set(G_deid.nodes) & set(G_orig.nodes))} / {G_orig.number_of_nodes()}')

In [ ]:
def draw_network(G: nx.Graph, ax, title: str, highlight_tags: bool = False):
    if G.number_of_nodes() == 0:
        ax.text(0.5, 0.5, 'empty graph', ha='center', va='center', color='#6f6f6f')
        return

    # Keep only the top-30 nodes by degree for readability
    top = sorted(G.degree, key=lambda x: x[1], reverse=True)[:30]
    sub = G.subgraph([n for n, _ in top])

    pos    = nx.spring_layout(sub, seed=42, k=0.8)
    weights = [sub[u][v]['weight'] for u, v in sub.edges()]
    max_w  = max(weights) if weights else 1
    widths = [1 + 3 * (w / max_w) for w in weights]

    tag_nodes  = [n for n in sub.nodes() if n.startswith('[')]
    word_nodes = [n for n in sub.nodes() if not n.startswith('[')]

    nx.draw_networkx_nodes(sub, pos, nodelist=word_nodes,
                           node_color='#78a9ff', node_size=200, ax=ax)
    if tag_nodes:
        nx.draw_networkx_nodes(sub, pos, nodelist=tag_nodes,
                               node_color='#f1c21b', node_size=300, ax=ax)
    nx.draw_networkx_edges(sub, pos, width=widths,
                           edge_color='#525252', ax=ax)
    nx.draw_networkx_labels(sub, pos, font_size=7,
                            font_color='#f4f4f4', ax=ax)
    ax.set_title(title, color='#f4f4f4', fontsize=11)
    ax.axis('off')


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
draw_network(G_orig, ax1, 'Original transcript')
draw_network(G_deid, ax2, 'De-identified transcript',  highlight_tags=True)

legend = [
    mpatches.Patch(color='#78a9ff', label='content word'),
    mpatches.Patch(color='#f1c21b', label='[TAG] replacement'),
]
ax2.legend(handles=legend, loc='lower right', framealpha=0.2,
           labelcolor='#f4f4f4', facecolor='#262626')

plt.suptitle('Word co-occurrence networks — before vs after de-identification',
             color='#f4f4f4', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Shared edges (relationships that survive)
shared = set(frozenset(e) for e in G_orig.edges()) & set(frozenset(e) for e in G_deid.edges())
print(f'\nEdges preserved: {len(shared)} / {G_orig.number_of_edges()} '
      f'({100*len(shared)/max(G_orig.number_of_edges(),1):.0f}%)')

---
## Step 5 — Word cloud: what remains visible?

A word cloud visualises the most frequent content words.

After de-identification: names disappear, but **concepts, actions, and domain terms**  
should still dominate — showing that the semantic content of the transcript is preserved.

In [ ]:
def make_wordcloud(text: str, title: str, ax, extra_stopwords: set = None):
    stops = STOPWORDS | (extra_stopwords or set())
    wc = WordCloud(
        width=700, height=400,
        background_color='#161616',
        colormap='Blues',
        stopwords=stops,
        min_word_length=3,
        max_words=60,
        prefer_horizontal=0.9,
    ).generate(text.lower())
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, color='#f4f4f4', fontsize=11)
    ax.axis('off')


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
make_wordcloud(text,      'Original transcript',       ax1)
make_wordcloud(text_deid, 'De-identified transcript',  ax2)
plt.suptitle('Word clouds — what the reader sees',
             color='#f4f4f4', fontsize=12)
plt.tight_layout()
plt.show()

---
## Step 6 — Save de-identified transcript

Save the de-identified text alongside a JSON log of all replacements  
— the log is your audit trail, equivalent to MaskAnyone's processing log for video.

In [ ]:
OUT_DIR = Path('text_deid_outputs')
OUT_DIR.mkdir(exist_ok=True)

(OUT_DIR / 'transcript_deidentified.txt').write_text(text_deid, encoding='utf-8')

full_log = {
    'tool':      'spaCy en_core_web_sm + regex',
    'ner_replacements':     ner_log,
    'pattern_replacements': pattern_log,
    'stats': {
        'original_words':    len(text.split()),
        'deid_words':        len(text_deid.split()),
        'entities_replaced': len(ner_log) + len(pattern_log),
    }
}
(OUT_DIR / 'deid_log.json').write_text(
    json.dumps(full_log, indent=2, ensure_ascii=False), encoding='utf-8'
)

print('Saved:')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size} bytes)')

---
## What did de-identification preserve?

| Feature | Preserved? | Notes |
|---------|-----------|-------|
| Topic/domain vocabulary | ✅ Yes | gesture, design, masking, prototype — still visible |
| Sentence structure / syntax | ✅ Yes | grammatical relationships intact |
| Speaker turns / dialogue flow | ✅ Yes | Facilitator / Participant A structure survives |
| Co-occurrence relationships | ✅ Mostly | Network structure largely preserved |
| Speaker identity | ❌ No | Replaced with [PERSON_1], [PERSON_2] |
| Institution affiliations | ❌ No | Replaced with [ORG_1], [ORG_2] |
| Geographic identifiers | ❌ No | Replaced with [PLACE_1] |
| Contact details | ❌ No | Replaced with [EMAIL] |

**Key insight:** thematic analysis, discourse analysis, and co-occurrence studies  
are largely unaffected by NER-based de-identification. Biographical or social  
network analysis (who knows whom?) is directly impacted — which is exactly  
the kind of re-identification risk you were trying to remove.

---
## Limitations

- **spaCy's small model** (`en_core_web_sm`) misses rare or domain-specific names.  
  For higher recall: use `en_core_web_trf` (transformer-based) or a fine-tuned model.
- **Indirect identifiers** ("the only left-handed participant", "the April cohort") are  
  not caught by NER. Manual review is still required.
- **Non-English text:** load the appropriate spaCy model (`nl_core_news_sm` for Dutch, etc.).
- **Re-identification from context:** even with all names removed, a small dataset with  
  distinctive events may still allow re-identification. Consult your data steward.

---
## References

- Owoyele et al. (2026). MaskingOPS. *Behavior Research Methods*. (under review)
- Honnibal & Montani (2017). spaCy 2: Natural language understanding with Bloom embeddings.
- Névéol et al. (2020). Clinical NLP for clinical NLP — overview of i2b2 de-id track.  
- Utrecht University Data Privacy Handbook — Pseudonymisation & Anonymisation.  
  [utrechtuniversity.github.io/dataprivacyhandbook](https://utrechtuniversity.github.io/dataprivacyhandbook)